# ML-10 — Content Action Playbook

The model from ML-08/ML-09 produces a score. A score is not an action. This notebook turns the score
into a queue a content editor can work through on a Monday morning, with a stated reason for every row
and explicit limits on what the queue is allowed to decide.


## 1. Ranked actions + reason codes

The queue is ordered by predicted risk, but risk alone is not enough to act on. Two pages with identical
scores can need opposite responses: one is a high-traffic page slipping from page one, the other is a
page nobody sees. So each row carries a **reason code** — the observable fact that put it in the queue —
and an **action** derived from that reason, not from the score.

Reason codes are deliberately readable. An editor who does not trust the model can still check whether
the reason is true, and act on the reason alone. That is the point: the model chooses the *order*, the
reason code justifies the *row*.

| Reason code | Condition | Suggested action |
|---|---|---|
| `HIGH_VALUE_AT_RISK` | Top-quartile impressions + high risk score | Review first — most clicks to lose |
| `SLIPPING_FROM_PAGE_1` | Position 8–15, high risk | Check intent match, internal links |
| `CTR_UNDER_CAPTURE` | Good position, CTR below its position band | Review title and meta description |
| `THIN_VISIBILITY` | Low impressions, high risk | Monitor only — not worth editorial time |
| `ENGAGEMENT_WEAK` | Traffic holding, scroll depth low | Review on-page content quality |
| `WATCH` | High risk, no distinguishing signal | Queue for the next review cycle |

`THIN_VISIBILITY` exists to *remove* rows from the work queue. A ranked list that sends editors to pages
with 40 impressions wastes the thing the list was supposed to save.


In [ ]:
# --- Setup: connect to the warehouse -------------------------------------
# Token comes from a Colab Secret named HF_TOKEN. Never paste it in a cell:
# this repo is public.
%pip -q install duckdb huggingface_hub scikit-learn matplotlib

import os, json, warnings
import numpy as np
import pandas as pd
import duckdb
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Confirm the release matches the documented build before trusting anything.
print(con.sql(f"SELECT COUNT(*) AS rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {FACT}").df())


In [ ]:
# --- Rebuild the monthly modelling frame ---------------------------------
# Grain: one row per content item per calendar month.
# Features come from the CURRENT month; the label comes from the NEXT month.
#
# Cached to /content/ (Colab local disk), NOT the repo — datasets must never
# enter git, and .gitignore + CI both block them.
CACHE = "/content/monthly_frame.parquet"

if os.path.exists(CACHE):
    frame = pd.read_parquet(CACHE)
    print("loaded from cache")
else:
    frame = con.sql(f"""
        WITH monthly AS (
            SELECT
                client_hash_id,
                content_hash_id,
                DATE_TRUNC('month', report_date)                  AS month,
                SUM(gsc_impressions)                              AS impressions,
                SUM(gsc_clicks)                                   AS clicks,
                -- NULLIF: gsc_avg_position = 0 means "no data", NOT rank zero.
                AVG(NULLIF(gsc_avg_position, 0))                  AS avg_position,
                SUM(scroll_events)                                AS scroll_events,
                SUM(sessions_ai)                                  AS sessions_ai
            FROM {FACT}
            GROUP BY 1, 2, 3
        ),
        with_next AS (
            SELECT *,
                   LEAD(impressions) OVER (
                       PARTITION BY client_hash_id, content_hash_id ORDER BY month
                   ) AS next_month_impressions
            FROM monthly
        )
        SELECT * FROM with_next
        WHERE next_month_impressions IS NOT NULL
          AND impressions > 0
    """).df()
    frame.to_parquet(CACHE)

# Derived columns.
frame["ctr"] = frame["clicks"] / frame["impressions"].replace(0, np.nan)
frame["ctr"] = frame["ctr"].fillna(0)

# Label: next month's impressions are at most 70% of this month's.
frame["decline_label"] = (frame["next_month_impressions"] <= 0.70 * frame["impressions"]).astype(int)

FEATURES = ["avg_position", "impressions", "ctr", "clicks", "scroll_events", "sessions_ai"]
frame = frame.dropna(subset=FEATURES)

print(f"rows: {len(frame):,}")
print(f"clients: {frame.client_hash_id.nunique()}  content items: {frame.content_hash_id.nunique():,}")
print(f"base rate (share labelled decline): {frame.decline_label.mean():.4f}")


In [ ]:
# --- Fit the model and score the held-out clients ------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

FEATURES = ["avg_position", "impressions", "ctr", "clicks", "scroll_events", "sessions_ai"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr, te = next(gss.split(frame[FEATURES], frame["decline_label"], groups=frame["client_hash_id"]))
train, test = frame.iloc[tr].copy(), frame.iloc[te].copy()

model = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced",
                               random_state=SEED, n_jobs=-1).fit(train[FEATURES],
                                                                 train["decline_label"])
test["risk_score"] = model.predict_proba(test[FEATURES])[:, 1]

BASE_RATE = float(test["decline_label"].mean())
AUC = float(roc_auc_score(test["decline_label"], test["risk_score"]))
print(f"held-out clients : {test.client_hash_id.nunique()}")
print(f"rows             : {len(test):,}")
print(f"base rate        : {BASE_RATE:.4f}")
print(f"AUC              : {AUC:.4f}")


In [ ]:
# --- Assign reason codes and actions -------------------------------------
# Thresholds are computed from the TRAINING data only. Deriving a cut-off from
# the test set would be a quiet form of leakage.
imp_q75    = train["impressions"].quantile(0.75)
imp_floor  = 100                       # below this, editorial time is wasted
scroll_q25 = train["scroll_events"].quantile(0.25)

# Expected CTR per integer position band, pooled (total clicks / total impressions).
# Pooled, not mean-of-ratios: a 2-impression page must not vote as loudly as a
# 200,000-impression page.
train["pos_band"] = train["avg_position"].round().clip(1, 50)
ctr_curve = (train.groupby("pos_band")
                  .apply(lambda g: g["clicks"].sum() / max(g["impressions"].sum(), 1))
                  .rename("expected_ctr"))

test["pos_band"] = test["avg_position"].round().clip(1, 50)
test = test.merge(ctr_curve, on="pos_band", how="left")
test["expected_ctr"] = test["expected_ctr"].fillna(train["clicks"].sum() / train["impressions"].sum())
test["ctr_gap"] = test["expected_ctr"] - test["ctr"]

HIGH_RISK = 0.60

def reason_code(r):
    if r.impressions < imp_floor:
        return "THIN_VISIBILITY"
    if r.impressions >= imp_q75 and r.risk_score >= HIGH_RISK:
        return "HIGH_VALUE_AT_RISK"
    if 8 <= r.avg_position <= 15 and r.risk_score >= HIGH_RISK:
        return "SLIPPING_FROM_PAGE_1"
    if r.avg_position <= 20 and r.ctr_gap > 0:
        return "CTR_UNDER_CAPTURE"
    if r.scroll_events <= scroll_q25 and r.risk_score >= HIGH_RISK:
        return "ENGAGEMENT_WEAK"
    return "WATCH"

ACTIONS = {
    "HIGH_VALUE_AT_RISK":  "Review first — highest measured click exposure",
    "SLIPPING_FROM_PAGE_1":"Check search-intent match and internal links",
    "CTR_UNDER_CAPTURE":   "Review title and meta description",
    "THIN_VISIBILITY":     "Monitor only — do not spend editorial time",
    "ENGAGEMENT_WEAK":     "Review on-page content quality",
    "WATCH":               "Queue for next review cycle",
}

test["reason_code"] = test.apply(reason_code, axis=1)
test["action"] = test["reason_code"].map(ACTIONS)

# Rank by expected clicks at risk, not by probability alone: a 0.9 score on a
# 200-impression page matters less than a 0.6 score on a 40,000-impression page.
test["clicks_at_risk"] = test["risk_score"] * test["clicks"]

queue = (test[test.reason_code != "THIN_VISIBILITY"]
           .sort_values("clicks_at_risk", ascending=False)
           .reset_index(drop=True))
queue.index += 1

print("reason code mix (full held-out set):")
print(test["reason_code"].value_counts().to_string(), "\n")
print(f"actionable queue: {len(queue):,} rows "
      f"({len(queue)/len(test):.1%} of held-out rows)\n")

cols = ["content_hash_id", "reason_code", "action", "risk_score",
        "impressions", "clicks", "avg_position", "ctr", "clicks_at_risk"]
print("TOP 20 — the queue an editor actually receives:")
print(queue.head(20)[cols].round(3).to_string())


## 2. Intended use and limits

**Who uses this.** A content editor or SEO lead deciding which pages get reviewed in a given cycle, when
there is capacity for tens of pages and hundreds of thousands are eligible.

**What it is for.** Ordering a review backlog. The queue answers *"which page do I open first?"* — not
*"is this page bad?"* and not *"what will happen to this page?"*

**Where it stops being valid.**

- **It is not a forecast.** The label is a measured association between this month's signals and next
  month's impressions. A page near the top of the queue is not scheduled to decline.
- **It is not causal.** Nothing here supports "fix X and traffic returns." Position leads the feature
  importances, but that describes how the fitted model used the column, not a lever.
- **It says nothing about Google's algorithm.** The features are performance measurements, not ranking factors.
- **New pages are out of scope.** A content item needs a prior month of history to be scored at all.
- **Below the volume floor it is noise.** A page moving from 4 impressions to 1 satisfies the decline
  definition and means nothing.
- **Client-held-out, not future-held-out.** The evaluation shows generalisation to unseen clients. It does
  not simulate deployment forward in time, which is the stronger test I did not run.


In [ ]:
# --- Scope card: what this queue does and does not cover -----------------
scope = {
    "unit_of_analysis":     "one content item in one calendar month",
    "held_out_clients":     int(test.client_hash_id.nunique()),
    "scored_rows":          int(len(test)),
    "actionable_rows":      int(len(queue)),
    "excluded_thin":        int((test.reason_code == "THIN_VISIBILITY").sum()),
    "volume_floor":         imp_floor,
    "base_rate":            round(BASE_RATE, 4),
    "auc":                  round(AUC, 4),
    "label_definition":     "next-month impressions <= 70% of current month",
    "validation":           "client-grouped holdout, no client in both sets",
    "claim_strength":       "observed / measured / directional / decision-support",
}
for k, v in scope.items():
    print(f"{k:22s} {v}")


## 3. Human review + the no-go list

**A person must check, before acting on any row:**

1. **Is the page still live and still meant to rank?** Retired, seasonal, and intentionally deindexed
   pages look identical to declining ones in this data.
2. **Was there a known cause?** A site migration, a redirect, a deliberate rewrite, or a de-indexing all
   produce the decline pattern and need no editorial review.
3. **Does the reason code hold up on inspection?** If the code says `CTR_UNDER_CAPTURE`, open the page and
   confirm the title and description are actually weak before rewriting anything.
4. **Is the volume worth the hour?** `clicks_at_risk` is the honest triage number, not `risk_score`.
5. **Is seasonality the whole story?** A monthly comparison cannot distinguish decline from an annual cycle.

**The no-go list — never automated, regardless of score:**

- **Deleting, pruning, or deindexing a page.** The model measures an impressions drop. It has no view of
  business value, conversions, legal or compliance need, or internal linking structure.
- **Publishing a rewrite without human review.** A high score is not a content brief.
- **Changing titles or meta descriptions in bulk.** `CTR_UNDER_CAPTURE` is a hypothesis to check, not a defect.
- **Reporting a queue position to a client as a prediction.** The framing must stay decision-support.
- **Acting on any row below the volume floor.** Included as `THIN_VISIBILITY` for completeness, excluded
  from the queue on purpose.
- **Using this to evaluate a writer or a vendor.** The model has no view of content quality.


In [ ]:
# --- Guardrails: quantify what the queue is NOT allowed to decide --------
guard = pd.DataFrame([
    {"guardrail": "Volume floor enforced",
     "check": f"rows below {imp_floor} impressions excluded from queue",
     "count": int((test.reason_code == "THIN_VISIBILITY").sum())},
    {"guardrail": "No page-level automation",
     "check": "every queue row requires human confirmation of its reason code",
     "count": int(len(queue))},
    {"guardrail": "Ranked by exposure, not probability",
     "check": "queue ordered by clicks_at_risk = risk_score * clicks",
     "count": int(len(queue))},
    {"guardrail": "Thresholds fitted on training data only",
     "check": "imp_q75 / scroll_q25 / CTR curve derived from train split",
     "count": int(len(train))},
])
print(guard.to_string(index=False))

# A high score on a near-zero-traffic page is the failure mode to watch for.
noise = test[(test.risk_score >= 0.9) & (test.impressions < imp_floor)]
print(f"\nHigh-score rows below the volume floor: {len(noise):,}")
print("These would have polluted the top of the queue without the floor —")
print("this is the 'ignoring low-volume noise' mistake, measured.")


## 4. Monitoring / retrain triggers

The queue goes stale silently — it keeps producing confident-looking rows long after it stopped being
right. So the triggers are numeric and checked on a schedule, not by feel.

| Trigger | Threshold | Why |
|---|---|---|
| **Base-rate drift** | monthly decline rate moves more than 5pp from training | The thing being predicted changed |
| **Precision@100 falls** | below the base rate + 10pp | The ranking has stopped beating chance |
| **Feature drift** | PSI above 0.25 on any feature | Inputs no longer resemble training data |
| **Coverage collapse** | actionable rows drop more than 30% | Reason-code thresholds have gone stale |
| **Calendar** | 3 months, unconditionally | Search behaviour drifts even when metrics do not |

Retrain on the two months preceding the trigger, then re-run the ML-09 audit before shipping. A retrained
model that has not been re-audited is not an improvement, just a newer number.


In [ ]:
# --- Compute the trigger values for this run -----------------------------
def psi(expected, actual, bins=10):
    """Population Stability Index. >0.25 = the input distribution has moved."""
    qs = np.linspace(0, 1, bins + 1)
    cuts = np.unique(np.quantile(expected, qs))
    if len(cuts) < 3:
        return 0.0
    e = np.histogram(expected, bins=cuts)[0] / len(expected)
    a = np.histogram(actual,   bins=cuts)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    top = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top].mean())

p100 = precision_at_k(test["decline_label"], test["risk_score"], 100)

triggers = pd.DataFrame([
    {"trigger": "base-rate drift", "current": round(BASE_RATE, 4),
     "threshold": f"±0.05 from {train.decline_label.mean():.4f}",
     "fires": abs(BASE_RATE - train.decline_label.mean()) > 0.05},
    {"trigger": "precision@100", "current": round(p100, 4),
     "threshold": f">= {BASE_RATE + 0.10:.4f}", "fires": p100 < BASE_RATE + 0.10},
    {"trigger": "coverage", "current": round(len(queue) / len(test), 4),
     "threshold": ">= 0.10 of scored rows", "fires": len(queue) / len(test) < 0.10},
])
print(triggers.to_string(index=False), "\n")

print("feature drift (train vs held-out clients):")
drift = pd.DataFrame([{"feature": f, "psi": round(psi(train[f].values, test[f].values), 4),
                       "fires": psi(train[f].values, test[f].values) > 0.25}
                      for f in FEATURES])
print(drift.to_string(index=False))
print("\nNote: PSI here measures difference BETWEEN CLIENTS, not over time.")
print("In production the same check runs train-month vs current-month.")


## 5. Exports for the paper

**Rule that matters here: no CSV.** The repo's CI runs a data-leak check that fails the build if any
committed `.csv` is not one of two allow-listed starter files. `work/**/*.csv` is gitignored as well, so
a queue export would either vanish or break the build.

So exports are:

- **`work/outputs/*.json`** — metrics. These are meant to be committed: they are the receipts my paper's
  numbers trace back to.
- **`work/outputs/figures/*.svg`** — the charts my deployed paper embeds. The capstone brief requires
  charts in the Results section.
- **The top-20 queue as a markdown table**, printed here and pasted into the paper. Small summary tables
  belong in the write-up, not in git as data.


In [ ]:
# --- Export metrics and figures ------------------------------------------
import os, json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = "work/outputs"; FIG = f"{OUT}/figures"
os.makedirs(FIG, exist_ok=True)

# --- 1. Metrics JSON (COMMIT THIS — it is the receipt for the paper) -----
metrics = {
    "label_definition": "next-month impressions <= 70% of current month",
    "validation": "client-grouped holdout (GroupShuffleSplit, seed 42)",
    "held_out_clients": int(test.client_hash_id.nunique()),
    "test_rows": int(len(test)),
    "base_rate": round(BASE_RATE, 4),
    "auc": round(AUC, 4),
    "precision_at_100": round(precision_at_k(test["decline_label"], test["risk_score"], 100), 4),
    "precision_at_1000": round(precision_at_k(test["decline_label"], test["risk_score"], 1000), 4),
    "volume_floor": imp_floor,
    "actionable_rows": int(len(queue)),
    "reason_code_mix": test["reason_code"].value_counts().to_dict(),
    "feature_importance": dict(zip(FEATURES, [round(float(v), 4)
                                              for v in model.feature_importances_])),
    "seed": SEED,
}
with open(f"{OUT}/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"wrote {OUT}/capstone_metrics.json")

# --- 2. Figure: model vs transparent baselines ---------------------------
from sklearn.metrics import roc_auc_score
bl = {"rank by worse position": test["avg_position"].values,
      "rank by low CTR": -test["ctr"].values,
      "Random Forest": test["risk_score"].values}
names = list(bl); vals = [precision_at_k(test["decline_label"], bl[n], 1000) for n in names]

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(names, vals, color=["#94a3b8", "#94a3b8", "#2563eb"])
ax.axvline(BASE_RATE, color="#dc2626", ls="--", lw=1.5,
           label=f"base rate {BASE_RATE:.2f}")
ax.set_xlabel("precision@1000 (higher is better)")
ax.set_title("Ranking quality vs transparent baselines")
ax.legend(); fig.tight_layout()
fig.savefig(f"{FIG}/model_vs_baseline.svg", format="svg", bbox_inches="tight")
print(f"wrote {FIG}/model_vs_baseline.svg")

# --- 3. Figure: feature importance ---------------------------------------
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(imp.index, imp.values, color="#2563eb")
ax.set_xlabel("Random Forest feature importance (descriptive, not causal)")
ax.set_title("What the fitted model leaned on")
fig.tight_layout()
fig.savefig(f"{FIG}/feature_importance.svg", format="svg", bbox_inches="tight")
print(f"wrote {FIG}/feature_importance.svg")

# --- 4. Top 20 queue as markdown (paste into the paper, do NOT commit CSV)
top = queue.head(20)[["reason_code", "action", "risk_score",
                      "impressions", "clicks", "avg_position"]].round(3)
print("\n--- paste into the paper's Recommendations section ---\n")
print(top.to_markdown())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
